# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SymbolPamnani/Flyrank-ML-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. Answer

The baseline is a simple transparent ranking score for prioritizing content items for review.

The goal is not to predict Google's ranking algorithm or prove that a page must be refreshed. The score is a decision-support baseline that identifies pages showing stronger directional signals associated with declining search performance.

The rule uses only signals that are available from the current content metadata and deliberately avoids the fields that directly define the decline proxy or use the recent trend windows.

The baseline combines three signals:

1. Search demand: higher search volume gives a page more potential search visibility impact.
2. Content freshness: pages that have gone longer since their last update receive more priority.
3. Content age: older content receives a small additional priority because it may warrant review.

Each signal is converted to a percentile rank so that the components are on a comparable scale.

Baseline score:

- 50% search-volume priority
- 30% freshness priority
- 20% content-age priority

Reason codes explain which signals contributed most to each item's score.

In [1]:
import pandas as pd
import numpy as np

DATA_PATH = "content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("\nColumns used by baseline:")
print([
    "search_volume",
    "days_since_last_update",
    "content_age_days"
])

print("\nMissing values:")
print(
    df[
        [
            "search_volume",
            "days_since_last_update",
            "content_age_days"
        ]
    ].isna().sum()
)

Shape: (30000, 44)

Columns used by baseline:
['search_volume', 'days_since_last_update', 'content_age_days']

Missing values:
search_volume             2468
days_since_last_update       0
content_age_days             0
dtype: int64


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## 2. Answer

The baseline score is calculated from three transparent components.

Search volume is ranked so that pages with greater search demand receive higher priority.

Days since last update is ranked so that less recently updated pages receive higher priority.

Content age is ranked so that older pages receive a smaller additional priority.

Missing search-volume values are assigned a neutral percentile rather than being treated as zero. The resulting score is used only for prioritization and is not interpreted as a probability.

The output contains the content identifier, baseline score, reason codes, and the main signals used to construct the ranking.

In [2]:
import os

os.makedirs("work/outputs", exist_ok=True)

baseline = df[
    [
        "content_id",
        "search_volume",
        "days_since_last_update",
        "content_age_days"
    ]
].copy()

# Percentile ranks.
# Missing search volume receives a neutral rank.
baseline["search_volume_rank"] = (
    baseline["search_volume"]
    .rank(pct=True)
    .fillna(0.5)
)

baseline["freshness_rank"] = (
    baseline["days_since_last_update"]
    .rank(pct=True)
)

baseline["age_rank"] = (
    baseline["content_age_days"]
    .rank(pct=True)
)

# Transparent weighted baseline score.
baseline["baseline_score"] = (
    0.50 * baseline["search_volume_rank"]
    + 0.30 * baseline["freshness_rank"]
    + 0.20 * baseline["age_rank"]
)

# Reason codes based on the strongest contributing signals.
def make_reason_codes(row):
    contributions = {
        "HIGH_SEARCH_DEMAND": 0.50 * row["search_volume_rank"],
        "STALE_CONTENT": 0.30 * row["freshness_rank"],
        "OLDER_CONTENT": 0.20 * row["age_rank"],
    }

    ordered = sorted(
        contributions,
        key=contributions.get,
        reverse=True
    )

    return ", ".join(ordered[:2])


baseline["reason_codes"] = baseline.apply(
    make_reason_codes,
    axis=1
)

# Rank highest priority first.
baseline = baseline.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

baseline["baseline_rank"] = baseline.index + 1

# Save queue.
output_path = "work/outputs/baseline_action_score.csv"

baseline.to_csv(
    output_path,
    index=False
)

print("Baseline queue created.")
print("Rows:", len(baseline))
print("Output:", output_path)

print("\nTop 10:")
print(
    baseline[
        [
            "baseline_rank",
            "content_id",
            "baseline_score",
            "reason_codes"
        ]
    ].head(10)
)

Baseline queue created.
Rows: 30000
Output: work/outputs/baseline_action_score.csv

Top 10:
   baseline_rank            content_id  baseline_score  \
0              1  content_5c5fab9d41e7        0.951226   
1              2  content_d9fd8bb80909        0.950345   
2              3  content_b40e32d5df10        0.950089   
3              4  content_a1defa0ee34a        0.948620   
4              5  content_4aeb7ad6d5d3        0.948518   
5              6  content_b954bc4acaad        0.947875   
6              7  content_1edbc8c5de7a        0.946876   
7              8  content_e6955a2c59dc        0.946739   
8              9  content_3d8dbd98077e        0.945740   
9             10  content_e6ac4ef9f617        0.945740   

                        reason_codes  
0  HIGH_SEARCH_DEMAND, STALE_CONTENT  
1  HIGH_SEARCH_DEMAND, STALE_CONTENT  
2  HIGH_SEARCH_DEMAND, STALE_CONTENT  
3  HIGH_SEARCH_DEMAND, STALE_CONTENT  
4  HIGH_SEARCH_DEMAND, STALE_CONTENT  
5  HIGH_SEARCH_DEMAND, STALE_CONTEN

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 3.Answer

The top 20 items are treated as a review queue rather than automatic refresh decisions.

For each item, the baseline provides:

- an action: review for possible refresh;
- reason codes explaining the strongest baseline signals;
- a confidence note based on how strongly the item ranks in the queue;
- a statement of what could make the recommendation wrong.

The baseline score does not establish that a page is declining or that refreshing it will improve performance. Human review is required before taking action.

In [3]:
top20 = baseline.head(20).copy()

top20["action"] = "REVIEW_FOR_REFRESH"

top20["confidence_note"] = np.select(
    [
        top20["baseline_score"] >= baseline["baseline_score"].quantile(0.99),
        top20["baseline_score"] >= baseline["baseline_score"].quantile(0.95),
    ],
    [
        "High baseline priority",
        "Moderate baseline priority",
    ],
    default="Lower baseline priority within top-20 queue"
)

top20["what_could_make_it_wrong"] = (
    "The page may not need a refresh; "
    "the score is based on metadata signals and requires human review."
)

review_columns = [
    "baseline_rank",
    "content_id",
    "baseline_score",
    "reason_codes",
    "action",
    "confidence_note",
    "what_could_make_it_wrong"
]

print(top20[review_columns].to_string(index=False))

 baseline_rank           content_id  baseline_score                      reason_codes             action        confidence_note                                                                           what_could_make_it_wrong
             1 content_5c5fab9d41e7        0.951226 HIGH_SEARCH_DEMAND, STALE_CONTENT REVIEW_FOR_REFRESH High baseline priority The page may not need a refresh; the score is based on metadata signals and requires human review.
             2 content_d9fd8bb80909        0.950345 HIGH_SEARCH_DEMAND, STALE_CONTENT REVIEW_FOR_REFRESH High baseline priority The page may not need a refresh; the score is based on metadata signals and requires human review.
             3 content_b40e32d5df10        0.950089 HIGH_SEARCH_DEMAND, STALE_CONTENT REVIEW_FOR_REFRESH High baseline priority The page may not need a refresh; the score is based on metadata signals and requires human review.
             4 content_a1defa0ee34a        0.948620 HIGH_SEARCH_DEMAND, STALE_CONTENT REVIEW

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## 4. Answer

A weak pick is an item that receives a high baseline rank even though the available evidence gives a limited reason to prioritize it.

Because this is a simple metadata-based baseline, high-ranked items can still be false positives. The queue is therefore intended for human investigation rather than automatic action.

The leakage check confirms that the baseline does not use:

- trend_direction
- trend_pct
- last-30-day performance fields
- previous-30-day performance fields
- 90-day performance fields
- impression_tier
- position_tier
- content_id or client_id as model signals

These fields are excluded because they can directly encode the decline definition, overlap the outcome window, act as derived product flags, or identify/group the data rather than represent a usable predictive feature.

In [4]:
# Inspect relatively weak picks within the top-20 queue.
# These are not necessarily errors; they are candidates for human review.

weak_pick_candidates = top20[
    (top20["search_volume_rank"] < 0.50)
    & (top20["freshness_rank"] < 0.50)
].copy()

print("Weak-pick candidates in top 20:", len(weak_pick_candidates))

if len(weak_pick_candidates) > 0:
    print("\nCandidates:")
    print(
        weak_pick_candidates[
            [
                "baseline_rank",
                "content_id",
                "baseline_score",
                "search_volume_rank",
                "freshness_rank",
                "age_rank",
                "reason_codes"
            ]
        ].to_string(index=False)
    )
else:
    print(
        "No top-20 items simultaneously had below-median "
        "search-volume and freshness ranks."
    )


# Leakage check.
forbidden_columns = [
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "impression_tier",
    "position_tier",
    "content_id",
    "client_id",
]

baseline_feature_columns = [
    "search_volume",
    "days_since_last_update",
    "content_age_days",
]

leaked_columns = sorted(
    set(baseline_feature_columns).intersection(
        forbidden_columns
    )
)

print("\nBaseline feature columns:")
print(baseline_feature_columns)

print("\nForbidden columns used by baseline:")
print(leaked_columns)

if len(leaked_columns) == 0:
    print("\nLeakage check: PASS")
else:
    print("\nLeakage check: REVIEW REQUIRED")

Weak-pick candidates in top 20: 0
No top-20 items simultaneously had below-median search-volume and freshness ranks.

Baseline feature columns:
['search_volume', 'days_since_last_update', 'content_age_days']

Forbidden columns used by baseline:
[]

Leakage check: PASS


In [5]:
# Evaluate the baseline against the defined decline proxy.
# trend_direction is used only for evaluation, never for scoring.

target_lookup = df[["content_id"]].copy()
target_lookup["declining_proxy"] = (
    df["trend_direction"] == "down"
).astype(int)

baseline_eval = baseline.merge(
    target_lookup,
    on="content_id",
    how="left",
    validate="one_to_one"
)

def precision_at_k(scores, labels, k=50):
    top_k = np.argsort(-np.asarray(scores))[:k]
    return np.asarray(labels)[top_k].mean()

baseline_precision_50 = precision_at_k(
    baseline_eval["baseline_score"].values,
    baseline_eval["declining_proxy"].values,
    k=50
)

base_rate = baseline_eval["declining_proxy"].mean()

print(f"Baseline Precision@50: {baseline_precision_50:.3f}")
print(f"Declining proxy base rate: {base_rate:.3f}")
print(f"Declining proxy base rate (%): {base_rate * 100:.1f}%")

Baseline Precision@50: 0.420
Declining proxy base rate: 0.542
Declining proxy base rate (%): 54.2%


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.